# train

In [ ]:
!python scripts/make_stage1_init_weight.py \
--cldm_config configs/model/cldm_stage2.yaml \
--sd_weight pretrained/v2-1_512-ema-pruned.ckpt \
--output pretrained/init_stage1.ckpt

In [ ]:
!python scripts/make_file_list.py \
    --hr toy_datasets/synthetic/train/hr_256 toy_datasets/synthetic/val/hr_256 \
    --ref toy_datasets/synthetic/train/ref_256 toy_datasets/synthetic/val/ref_256 \
    --val-size 5 \
    --save-folder filelists

In [ ]:
!python train.py \
--config configs/train_cldm_stage1.yaml

In [ ]:
!python scripts/make_stage2_init_weight.py \
--cldm_config configs/model/cldm_stage2.yaml \
--sd_weight save_dir/stage1/lightning_logs/version_7/checkpoints/step=5-val_loss=1.1529.ckpt \
--hyper_encoder_weight save_dir/stage1/lightning_logs/version_7/checkpoints/step=5-val_loss=1.1529.ckpt \
--output pretrained/init_stage2.ckpt

In [ ]:
!python train.py \
--config configs/train_cldm_stage2.yaml

# inference

In [ ]:
# whole
!python inference.py \
    --ckpt ./magc_ckpts/ckpts_stage2/v40_step=120999-lpips=0.3132.ckpt \
    --config configs/model/cldm_stage2.yaml \
    --input_path ./toy_datasets/real-world/val/ \
    --steps 50 \
    --batchsize 30 \
    --output_path ./metrics/ \
    --device cuda

In [ ]:
# tile test (command runs but not actually wired)
!python inference.py \
    --ckpt ./magc_ckpts/ckpts_stage2/v40_step=120999-lpips=0.3132.ckpt \
    --config configs/model/cldm_stage2.yaml \
    --input_path ./toy_datasets/real-world/val/ \
    --steps 50 \
    --batchsize 30 \
    --output_path ./metrics/ \
    --device cuda \
    --tiled \
    --tile_size 128 \
    --tile_stride 256

In [ ]:
# warm up test
!python inference.py \
    --ckpt ./magc_ckpts/ckpts_stage2/v40_step=120999-lpips=0.3132.ckpt \
    --config configs/model/cldm_stage2.yaml \
    --input_path ./toy_datasets/real-world/val/ \
    --steps 1 \
    --batchsize 1 \
    --output_path ./metrics-warmup/ \
    --device cuda

In [ ]:
# inference test from custom trained model
!python inference.py \
    --ckpt save_dir/stage2/lightning_logs/version_0/checkpoints/step=5-lpips=0.9901.ckpt \
    --config configs/model/cldm_stage2.yaml \
    --input_path ./toy_datasets/real-world/val/ \
    --steps 50 \
    --batchsize 30 \
    --output_path ./metrics/ \
    --device cuda